In [1]:
import sys
from pathlib import Path

sys.path.append("..")

import torch
from lightning import seed_everything
from src import Module
from src.constants import DEFAULT_SEED
from src.graph import KNNGraph
from src.transforms import LineGraph
from torch_geometric.loader import NeighborLoader

/home/ghuynh/THESE/CLASSIFICATION/Lightning-CEGANN2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BATCH_SIZE = 16
CKPT = "../lightning_logs/version_45927073/checkpoints/epoch=101-step=131378.ckpt"

In [3]:
_ = seed_everything(DEFAULT_SEED, verbose=False)

In [4]:
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
k = ckpt["datamodule_hyper_parameters"]["k"]
ckpt_params = ckpt["hyper_parameters"] | ckpt["datamodule_hyper_parameters"]
k

14

In [5]:
from ase.io import read

structure = Path().absolute().parent / "data" / "test" / "raw" / "POSCAR_CUBIC"
atoms = read(structure)

In [6]:
knn = KNNGraph(k=k)

data = knn.convert(atoms)
lg_data = LineGraph().forward(data)

num_nodes = lg_data.num_nodes
if num_nodes is None:
    raise ValueError("The number of nodes in the graph is undefined.")

In [7]:
model = Module.load_from_checkpoint(CKPT, weights_only=False, **ckpt_params)
model = model.eval()

In [8]:
num_layers = ckpt_params["model_kwargs"]["n_bond_conv"]
loader = NeighborLoader(
    lg_data,
    num_neighbors=[k] * num_layers,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

In [9]:
with torch.inference_mode():
    # for batch in loader:
    out = model(lg_data)

In [10]:
preds = out.argmax(dim=1)

In [11]:
lg_data.bond_source.unique(return_counts=True)

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11], device='cuda:0'),
 tensor([14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14], device='cuda:0'))

In [12]:
lg_data.bond_target.unique(return_counts=True)

(tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11], device='cuda:0'),
 tensor([10, 14, 17, 13, 14, 16, 15, 13, 17, 14, 13, 12], device='cuda:0'))